# EXP-007 - Confirmatory: Temporal-Stability Feature Selection

**Not a hypothesis** - confirmatory replication of the winners' validation/selection
discipline (`docs/kaggle/validation-and-selection-playbook.md`). Isolates the *selection*
step by starting from EXP-006's exact feature engine and only removing features.

**The motivating result (EXP-006):** the aggregation engine lifted the holdout +0.0122
(p=5.6e-25) but the private LB only +0.0046, and the CV-LB gap *widened* to 0.0343 - more
features made the training-period overfit worse. This experiment tests the fix.

**Selection applied (in order):**
1. train/test screening - drop raw high-cardinality ids `card1,card2,card3,card5,addr1,addr2`
   (keep their frequency encodings)
2. adversarial validation - fit a train-vs-test model, report AUC + top drifters (diagnostic)
3. time-consistency filter - drop features whose signed univariate AUC flips sign or decays
   between the first and last training month
4. seen/unseen-UID holdout segmentation (diagnostic)

**Success signature:** private LB held/raised with a *smaller* CV-LB gap, and the gain
concentrated on unseen UIDs. Same LightGBM/params as EXP-001..006.

**Outputs:** `holdout_pred_exp007.csv`, `submission.csv`. DeLong off-notebook.

In [ ]:
import os
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SPLIT_QUANTILE = 0.8
SECONDS_PER_MONTH = 86400 * 30.44
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}
MISSING_TOKEN = "__missing__"
FREQ_NUMERIC_CATS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
RAW_IDS_TO_DROP = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
D_NORM_COLS = [f"D{i}" for i in range(1, 16) if i != 9]
GROUP_KEYS = ["card1_addr1", "card1_addr1_P_emaildomain", "uid"]
COARSE_KEYS = ["card1", "card1_addr1", "card1_addr1_P_emaildomain"]
COARSE_VALS = ["TransactionAmt", "D9", "D11"]
UID_MEANSTD_VALS = ["TransactionAmt", "D4", "D9", "D10", "D15"]
C_MEAN_COLS = [f"C{i}" for i in range(1, 15) if i != 3]
M_COLS = [f"M{i}" for i in range(1, 10)]
NUNIQUE_COLS = ["P_emaildomain", "dist1", "DT_M", "id_02", "cents",
                "C13", "V314", "V127", "V136", "V309", "V307", "V320"]

LGB_PARAMS = dict(
    objective="binary", learning_rate=0.05, num_leaves=192, min_data_in_leaf=100,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=42, n_jobs=-1, verbosity=-1,
)
MAX_ROUNDS = 5000
ES_PATIENCE = 200

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not attached (Add Input -> Competitions).")
    DATA_DIR = hits[0].parent
else:
    DATA_DIR = Path("../../../data/raw")
print(f"Data dir: {DATA_DIR}")

## 0. Feature + selection functions - inline mirrors of `src/features/{engineering,selection}.py`

In [ ]:
def _as_str(values):
    return values.astype("object").where(values.notna(), MISSING_TOKEN).astype(str)

def frequency_encode(train_values, values):
    freq = _as_str(train_values).value_counts(normalize=True)
    return _as_str(values).map(freq).fillna(0.0).astype("float32")

def label_encode(train_values, values):
    cats = {v: i for i, v in enumerate(sorted(_as_str(train_values).unique()))}
    return _as_str(values).map(cats).fillna(-1).astype("int32")

def split_email_domain(values, prefix):
    parts = _as_str(values).str.split(".")
    return pd.DataFrame({f"{prefix}_provider": parts.str[0], f"{prefix}_suffix": parts.str[-1]}, index=values.index)

def build_categorical_block(train_df, df, label_cols, freq_cols):
    out = pd.DataFrame(index=df.index)
    for col in label_cols:
        out[f"{col}_le"] = label_encode(train_df[col], df[col])
    for col in freq_cols:
        out[f"{col}_freq"] = frequency_encode(train_df[col], df[col])
    return out

def add_time_features(dt):
    return pd.DataFrame({"tx_hour": ((dt // 3600) % 24).astype("int32"), "tx_dow": ((dt // 86400) % 7).astype("int32")}, index=dt.index)

def add_amount_features(amount):
    cents = (amount - np.floor(amount)).round(2)
    return pd.DataFrame({"amt_log1p": np.log1p(amount).astype("float32"), "amt_cents": cents.astype("float32")}, index=amount.index)

def normalize_d_columns(df, dt, d_cols):
    days = dt / 86400.0
    out = pd.DataFrame(index=df.index)
    for col in d_cols:
        out[f"{col}_norm"] = (df[col] - days).astype("float32")
    return out

def make_uid(df, card_col="card1", addr_col="addr1", dt_col="TransactionDT", d1_col="D1"):
    day = (df[dt_col] / 86400.0).round()
    ref = (day - df[d1_col]).round().astype("Int64").astype("string").fillna("NA")
    card = df[card_col].astype("Int64").astype("string").fillna("NA")
    addr = df[addr_col].astype("Int64").astype("string").fillna("NA")
    return (card + "_" + addr + "_" + ref).astype("object")

def combine_columns(df, c1, c2, sep="_"):
    return (_as_str(df[c1]) + sep + _as_str(df[c2])).astype("object")

def aggregate_group(df, key, value_cols, aggs=("mean", "std")):
    g = df.groupby(key); out = pd.DataFrame(index=df.index)
    for col in value_cols:
        for agg in aggs:
            out[f"{col}_{key}_{agg}"] = g[col].transform(agg).astype("float32")
    return out

def aggregate_nunique(df, key, value_cols):
    g = df.groupby(key); out = pd.DataFrame(index=df.index)
    for col in value_cols:
        out[f"{key}_{col}_ct"] = g[col].transform("nunique").astype("float32")
    return out

# --- selection (mirror of src/features/selection.py) ---
def signed_univariate_auc(y, x):
    x = np.asarray(x, dtype=float); y = np.asarray(y)
    mask = ~np.isnan(x)
    if mask.sum() < 2:
        return 0.5
    ym = y[mask]
    if len(np.unique(ym)) < 2:
        return 0.5
    return float(roc_auc_score(ym, x[mask]))

def time_consistency_filter(features, y, period, flip_margin=0.02, decay_strong=0.05, decay_weak=0.01):
    periods = sorted(set(np.asarray(period).tolist()))
    first = np.asarray(period) == periods[0]; last = np.asarray(period) == periods[-1]
    keep, dropped = [], {}
    for name, values in features.items():
        values = np.asarray(values, dtype=float)
        a0 = signed_univariate_auc(y[first], values[first])
        a1 = signed_univariate_auc(y[last], values[last])
        d0, d1 = a0 - 0.5, a1 - 0.5
        flip = (d0 > flip_margin and d1 < -flip_margin) or (d0 < -flip_margin and d1 > flip_margin)
        decay = (abs(d0) > decay_strong and abs(d1) < decay_weak) or (abs(d1) > decay_strong and abs(d0) < decay_weak)
        if flip: dropped[name] = (round(a0, 4), round(a1, 4), "flip")
        elif decay: dropped[name] = (round(a0, 4), round(a1, 4), "decay")
        else: keep.append(name)
    return keep, dropped

def seen_unseen_masks(entity, train_mask, eval_mask):
    entity = np.asarray(entity, dtype=object)
    train_entities = set(entity[train_mask].tolist())
    is_seen = np.array([e in train_entities for e in entity], dtype=bool)
    return eval_mask & is_seen, eval_mask & ~is_seen

## 1. Build EXP-006 feature engine over the union (identical to EXP-006)

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity
test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]
test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity
n_train = len(train)
full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
del train, test

full["card1_addr1"] = combine_columns(full, "card1", "addr1")
full["card1_addr1_P_emaildomain"] = combine_columns(full, "card1_addr1", "P_emaildomain")
full["uid"] = make_uid(full).to_numpy()
full["DT_M"] = (full["TransactionDT"] / SECONDS_PER_MONTH).astype(int)
full["cents"] = (full["TransactionAmt"] - np.floor(full["TransactionAmt"])).round(2).astype("float32")

m_num = {}
for c in M_COLS:
    codes, _ = pd.factorize(full[c])
    m_num[c] = np.where(full[c].isna().to_numpy(), np.nan, codes).astype("float32")
m_num = pd.DataFrame(m_num, index=full.index); m_num["uid"] = full["uid"].to_numpy()

agg_frames = []
for key in COARSE_KEYS:
    agg_frames.append(aggregate_group(full, key, COARSE_VALS, ("mean", "std")))
agg_frames.append(aggregate_group(full, "uid", UID_MEANSTD_VALS, ("mean", "std")))
agg_frames.append(aggregate_group(full, "uid", C_MEAN_COLS, ("mean",)))
agg_frames.append(aggregate_group(full, "uid", ["C14"], ("std",)))
agg_frames.append(aggregate_group(m_num, "uid", M_COLS, ("mean",)))
agg_frames.append(aggregate_nunique(full, "uid", NUNIQUE_COLS))
agg_frames.append(pd.DataFrame({"outsider15": (np.abs(full["D1"] - full["D15"]) > 3).astype("float32")}, index=full.index))
agg_engine = pd.concat(agg_frames, axis=1)
del agg_frames, m_num

for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:
    full = pd.concat([full, split_email_domain(full[col], prefix)], axis=1)

numeric_features = [c for c in full.columns if full[c].dtype != "O" and c not in EXCLUDE_COLS and c not in ("DT_M",)]
label_cols = [c for c in full.columns if full[c].dtype == "O" and c not in GROUP_KEYS]
freq_cols = label_cols + FREQ_NUMERIC_CATS + GROUP_KEYS

row_local = pd.concat([add_time_features(full["TransactionDT"]), add_amount_features(full["TransactionAmt"]),
                       normalize_d_columns(full, full["TransactionDT"], D_NORM_COLS), agg_engine], axis=1)
X_num_full = pd.concat([full[numeric_features].astype("float32"), row_local], axis=1)
cat_source_full = full[label_cols + FREQ_NUMERIC_CATS + GROUP_KEYS].copy()
y = full["isFraud"].to_numpy(); dt = full["TransactionDT"].to_numpy()
months = full["DT_M"].to_numpy(); trans_ids = full["TransactionID"].to_numpy(); uid_all = full["uid"].to_numpy()
del full, row_local, agg_engine

is_train = np.arange(len(X_num_full)) < n_train
cutoff = np.quantile(dt[is_train], SPLIT_QUANTILE)
train_mask = is_train & (dt < cutoff); holdout_mask = is_train & (dt >= cutoff)
print(f"Engine features: {X_num_full.shape[1]} | train {train_mask.sum():,} | holdout {holdout_mask.sum():,}")

def make_X(fit_mask, rows_mask, keep=None):
    block = build_categorical_block(cat_source_full[fit_mask], cat_source_full[rows_mask], label_cols, freq_cols)
    X = pd.concat([X_num_full[rows_mask].reset_index(drop=True), block.reset_index(drop=True)], axis=1)
    return X[keep] if keep is not None else X

## 2. Feature selection (screening -> adversarial -> time-consistency)

In [ ]:
# Build the train-partition matrix once; selection is computed here and applied everywhere.
X_tr_full = make_X(train_mask, train_mask)
all_cols = list(X_tr_full.columns)
y_tr = y[train_mask].astype(int)
tr_months = months[train_mask]

# (1) train/test screening: drop raw high-cardinality ids (freq encodings kept)
after_screen = [c for c in all_cols if c not in RAW_IDS_TO_DROP]
print(f"(1) screening dropped raw ids: {RAW_IDS_TO_DROP}")

# (2) adversarial validation (diagnostic): can a model tell train from test?
rng = np.random.default_rng(42)
samp = rng.choice(len(is_train), size=min(200000, len(is_train)), replace=False)
X_adv = make_X(train_mask, np.isin(np.arange(len(is_train)), samp), keep=after_screen)
adv_target = (~is_train)[samp].astype(int)
adv = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=64, seed=42, n_jobs=-1, verbosity=-1)
cut = int(0.8 * len(samp))
adv.fit(X_adv.iloc[:cut], adv_target[:cut])
adv_auc = roc_auc_score(adv_target[cut:], adv.predict_proba(X_adv.iloc[cut:])[:, 1])
adv_imp = pd.Series(adv.feature_importances_, index=X_adv.columns).sort_values(ascending=False)
print(f"(2) adversarial train-vs-test AUC: {adv_auc:.4f} (0.5 = indistinguishable; higher = more drift)")
print("    top 15 drifting features:"); print(adv_imp.head(15).to_string())
del X_adv

# (3) time-consistency filter: drop features that flip/decay across train months
feats = {c: X_tr_full[c].to_numpy() for c in after_screen}
keep_cols, dropped = time_consistency_filter(feats, y_tr, tr_months)
print(f"(3) time-consistency: kept {len(keep_cols)}, dropped {len(dropped)}")
for name, (a0, a1, reason) in list(dropped.items())[:20]:
    print(f"    drop {name}: auc_first={a0} auc_last={a1} ({reason})")
del X_tr_full, feats
print(f"\nFINAL feature count: {len(keep_cols)} (from {len(all_cols)})")

## 3. Scheme B GroupKFold + Scheme A model on the selected features

In [ ]:
train_months_all = months[is_train]
fold_aucs = []
t0 = time.time()
for m in sorted(set(train_months_all)):
    tr = is_train & (months != m); va = is_train & (months == m)
    X_tr = make_X(tr, tr, keep=keep_cols); X_va = make_X(tr, va, keep=keep_cols)
    clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
    clf.fit(X_tr, y[tr].astype(int), eval_set=[(X_va, y[va].astype(int))], eval_metric="auc",
            callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)])
    fold_aucs.append(roc_auc_score(y[va].astype(int), clf.predict_proba(X_va)[:, 1]))
    del X_tr, X_va
    print(f"fold month={m}: AUC={fold_aucs[-1]:.4f}  ({(time.time()-t0)/60:.1f} min)")
scheme_b_mean, scheme_b_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f"\nScheme B GroupKFold: {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")

In [ ]:
es_month = months[train_mask].max()
sub_tr = train_mask & (months < es_month); es_va = train_mask & (months == es_month)
X_sub_tr = make_X(train_mask, sub_tr, keep=keep_cols); X_es_va = make_X(train_mask, es_va, keep=keep_cols)
es_clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
es_clf.fit(X_sub_tr, y[sub_tr].astype(int), eval_set=[(X_es_va, y[es_va].astype(int))], eval_metric="auc",
           callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)])
final_rounds = max(int(es_clf.best_iteration_ * 1.1), 100)
del X_sub_tr, X_es_va
print(f"ES month: {es_month} | best_iter: {es_clf.best_iteration_} | refit: {final_rounds}")

X_train = make_X(train_mask, train_mask, keep=keep_cols)
model = lgb.LGBMClassifier(n_estimators=final_rounds, **LGB_PARAMS)
model.fit(X_train, y[train_mask].astype(int)); del X_train

X_holdout = make_X(train_mask, holdout_mask, keep=keep_cols)
val_proba = model.predict_proba(X_holdout)[:, 1]; del X_holdout
holdout_auc = roc_auc_score(y[holdout_mask].astype(int), val_proba)
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (EXP-006: 0.9421)")

# (4) seen/unseen-UID segmentation of the holdout
seen, unseen = seen_unseen_masks(uid_all, train_mask, holdout_mask)
hold_idx = np.where(holdout_mask)[0]
seen_h = np.isin(hold_idx, np.where(seen)[0]); unseen_h = ~seen_h
yv = y[holdout_mask].astype(int)
auc_seen = roc_auc_score(yv[seen_h], val_proba[seen_h]) if seen_h.sum() and len(np.unique(yv[seen_h]))>1 else float('nan')
auc_unseen = roc_auc_score(yv[unseen_h], val_proba[unseen_h]) if unseen_h.sum() and len(np.unique(yv[unseen_h]))>1 else float('nan')
print(f"(4) holdout by UID: seen={seen_h.sum():,} rows AUC={auc_seen:.4f} | unseen={unseen_h.sum():,} rows AUC={auc_unseen:.4f}")

## 4. Save holdout predictions and submission

In [ ]:
pd.DataFrame({"TransactionID": trans_ids[holdout_mask], "y_true": yv, "score": val_proba}).to_csv("holdout_pred_exp007.csv", index=False)
print("Saved holdout_pred_exp007.csv")

X_test = make_X(train_mask, ~is_train, keep=keep_cols)
test_proba = model.predict_proba(X_test)[:, 1]
pd.DataFrame({"TransactionID": trans_ids[~is_train].astype(int), "isFraud": test_proba}).to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({(~is_train).sum():,} rows)")

print("\n=== EXP-007 summary ===")
print(f"Selected features   : {len(keep_cols)} (dropped {len(RAW_IDS_TO_DROP)} raw ids + {len(dropped)} time-inconsistent)")
print(f"Adversarial AUC     : {adv_auc:.4f}")
print(f"Scheme A holdout AUC: {holdout_auc:.4f}  (EXP-006: 0.9421)")
print(f"Scheme B GroupKFold : {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}  (EXP-006: 0.9561)")
print(f"Holdout seen/unseen : {auc_seen:.4f} / {auc_unseen:.4f}")
print(f"Per-fold AUCs       : {[round(float(a), 4) for a in fold_aucs]}")